# Computational processes implemented with neural networks: the code behind the slides

AI 4200 · Deep learning. One section per process in the lecture, in lecture order: a GAN, a self-play DQN,
diffusion, a tiny NeRF, and the invention of writing from the brainstorm.

- **Runtime.** Everything runs on a CPU. A full run takes about an hour (the two NeRF runs are 15 minutes each).
  Set `QUICK = True` for a smoke test of a minute or two (the numbers then won't match the slides).
  Finished runs are cached in `cache/`, so an interrupted run picks up where it stopped.
- **Data.** The 10,000 MNIST digits from the `mnist` npm package (the same digits the slides used), downloaded below.
- **Reproducing the slides.** With `QUICK = False`, the printed numbers match the slides on a CPU.
  The experiments stay on the CPU on purpose, because a GPU draws different random numbers.
  The one exception is the diffusion CNN in section 3, which wants a GPU.
- To play the tic-tac-toe network, open the separate notebook `week11.2-dqn-tictactoe.ipynb` (the QR code on the slide).

In [ ]:
QUICK = False          # True: a few steps of everything, as a smoke test

import json
import math
import os
import random
import time
import urllib.request
from functools import lru_cache

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

BLUE, ORANGE, INK, GREY = "#2557D6", "#C2510F", "#16181D", "#8E94A0"
CACHE = "cache"
os.makedirs(CACHE, exist_ok=True)

# Optional, on Colab: keep the cache in Google Drive so it survives a disconnect.
# from google.colab import drive; drive.mount("/content/drive"); CACHE = "/content/drive/MyDrive/ai4200-week11.2-cache"


def cached(name, compute):
    """Run compute() once and keep its JSON result in the cache folder."""
    path = os.path.join(CACHE, name + ("-quick" if QUICK else "") + ".json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    result = compute()
    with open(path, "w") as f:
        json.dump(result, f)
    return result


def load_digits():
    """The 10,000 digits of the mnist npm package: pixels in [0, 1] (n x 784) and labels, ordered 0s then 1s ..."""
    os.makedirs("data", exist_ok=True)
    images, labels = [], []
    for digit in range(10):
        path = f"data/{digit}.json"
        if not os.path.exists(path):
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/cazala/mnist/master/src/digits/{digit}.json", path)
        with open(path) as f:
            pixels = np.array(json.load(f)["data"], dtype=np.float32).reshape(-1, 784)
        images.append(pixels)
        labels.append(np.full(len(pixels), digit))
    return torch.tensor(np.concatenate(images)), torch.tensor(np.concatenate(labels))


def show_images(images, titles, suptitle, cmap="gray_r", size=1.6):
    """One row of images with a title under each."""
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size + 0.6))
    for ax, image, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(suptitle)
    plt.show()


DIGITS, LABELS = load_digits()
print(DIGITS.shape, "digits")

## 1. Competition: a GAN on MNIST
Slides: *The forward pass*, *Results: noise becomes digits*. A generator G (random vector z, 64 numbers → image)
and a discriminator D (image → p̂ = Pr(real)) take turns: D steps on L_D = −log p̂(real) − log(1 − p̂(fake)),
then G steps through D with D's weights held fixed. 12,000 steps of 64 digits, about 5 minutes.

The slide writes the generator's loss in the minimax form, log(1 − p̂(fake)). This run minimizes −log p̂(fake)
(the "flipped label", as in the original paper's practical advice): it pushes in the same direction but has a
steep slope when D confidently rejects a fake, so G does not stall early. The real label is 0.9, not 1
(one-sided label smoothing), which keeps D from winning outright.

In [ ]:
GAN_Z = 64
GAN_BATCH = 64
GAN_STEPS = 300 if QUICK else 12_000
GAN_SNAPS = [0, 300, 2000, 6000, 12_000]
REAL_LABEL = 0.9
FORWARD_STEP = 300          # the step whose numbers the forward-pass slide shows


def gan_samples(generator, z_fixed):
    """Images for 16 fixed random vectors, pixels back in [0, 1]."""
    generator.eval()
    with torch.no_grad():
        images = generator(z_fixed).view(-1, 28, 28)
    generator.train()
    return ((images + 1) / 2).tolist()


def train_gan():
    """Train the MLP GAN; return sample grids, the forward-pass numbers at step 300 and the loss curves."""
    torch.manual_seed(0)
    np.random.seed(0)
    X = DIGITS * 2 - 1                        # tanh output range [-1, 1]
    generator = nn.Sequential(
        nn.Linear(GAN_Z, 256), nn.BatchNorm1d(256), nn.SiLU(),
        nn.Linear(256, 512), nn.BatchNorm1d(512), nn.SiLU(),
        nn.Linear(512, 784), nn.Tanh(),
    )
    discriminator = nn.Sequential(
        nn.Linear(784, 512), nn.SiLU(), nn.Dropout(0.3),
        nn.Linear(512, 256), nn.SiLU(), nn.Dropout(0.3),
        nn.Linear(256, 1), nn.Sigmoid(),
    )
    opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.999))
    bce = nn.BCELoss()
    z_fixed = torch.randn(16, GAN_Z)
    result = {"snaps": {"0": gan_samples(generator, z_fixed)}, "curve": []}
    real_examples = torch.randperm(len(X))[:16]
    result["real"] = ((X[real_examples] + 1) / 2).view(-1, 28, 28).tolist()

    for step in range(1, GAN_STEPS + 1):
        real = X[torch.randint(0, len(X), (GAN_BATCH,))]
        z = torch.randn(GAN_BATCH, GAN_Z)
        fake = generator(z)

        # discriminator step: fake.detach() keeps the generator out of it
        p_hat_real = discriminator(real)
        p_hat_fake = discriminator(fake.detach())
        loss_d = bce(p_hat_real, torch.full_like(p_hat_real, REAL_LABEL)) + bce(p_hat_fake, torch.zeros_like(p_hat_fake))
        opt_d.zero_grad()
        loss_d.backward()
        opt_d.step()

        # generator step: the gradient flows back through the discriminator, whose weights are not stepped
        p_hat_fake_for_g = discriminator(fake)
        loss_g = bce(p_hat_fake_for_g, torch.ones_like(p_hat_fake_for_g))
        opt_g.zero_grad()
        loss_g.backward()
        opt_g.step()

        if step == FORWARD_STEP:
            result["forward"] = {"z_first": z[0, :3].tolist(),
                                 "p_hat_real": p_hat_real[0].item(), "p_hat_fake": p_hat_fake[0].item()}
        if step % 100 == 0:
            result["curve"].append({"step": step, "p_real": p_hat_real.mean().item(), "p_fake": p_hat_fake.mean().item()})
        if step in GAN_SNAPS:
            result["snaps"][str(step)] = gan_samples(generator, z_fixed)
    return result


gan = cached("gan", train_gan)

**Forward pass (step 300).** One real digit and one fake from the batch.
The slide shows p̂(real) = 0.60, p̂(fake) = 0.24, L_D = 0.51 + 0.27 = 0.79 and the first three numbers of z
(the slide rounds each p̂ to two decimals before the arithmetic, and so does this cell).

In [ ]:
fwd = gan["forward"]
p_hat_real = round(fwd["p_hat_real"], 2)
p_hat_fake = round(fwd["p_hat_fake"], 2)
print("z starts      ", np.round(fwd["z_first"], 2))
print(f"p_hat(real) = {p_hat_real:.2f}   p_hat(fake) = {p_hat_fake:.2f}")
print(f"L_D = -log p_hat(real) - log(1 - p_hat(fake)) = {-math.log(p_hat_real):.2f} + {-math.log(1 - p_hat_fake):.2f}"
      f" = {-math.log(p_hat_real) - math.log(1 - p_hat_fake):.2f}")
print(f"L_G (slide's form) = log(1 - p_hat(fake)) = {math.log(1 - p_hat_fake):.2f}")

In [ ]:
steps = [s for s in ["0", "300", "2000", "6000", "12000"] if s in gan["snaps"]]
fig, axes = plt.subplots(4, len(steps) + 1, figsize=(1.5 * (len(steps) + 1), 6.4))
for col, step in enumerate(steps + ["real"]):
    images = gan["real"] if step == "real" else gan["snaps"][step]
    for row in range(4):
        axes[row, col].imshow(images[row], cmap="gray")
        axes[row, col].axis("off")
    axes[0, col].set_title("real digits" if step == "real" else f"step {int(step):,}", fontsize=9)
fig.suptitle("The same four random vectors z through the generator as it trains")
plt.show()

curve = gan["curve"]
plt.plot([c["step"] for c in curve], [c["p_real"] for c in curve], color=BLUE, label="p̂ on real digits")
plt.plot([c["step"] for c in curve], [c["p_fake"] for c in curve], color=ORANGE, label="p̂ on fakes")
plt.xlabel("step")
plt.ylabel("discriminator's p̂ = Pr(real), batch mean")
plt.title("Neither side wins: the game stays balanced")
plt.legend()
plt.show()

## 2. Self-play: a DQN learns tic-tac-toe
Slides: *The forward pass*, *The backward pass*, *Results: a random player learns the game*.
One Q-network plays both sides (the board is sign-flipped for O). For a move a from board s:
target q = r if the move ended the game, otherwise q = −γ · max Q̂(s′) from a frozen copy (O's best reply is X's worst),
loss L = (Q̂(s, a) − q)². 30,000 games, about 11 minutes.

The progress checks (every position with an open win or a single threat, games against a random and a perfect player)
draw random numbers between training games, so they are part of what makes the run reproducible.

In [ ]:
DQN_GAMES = 1_000 if QUICK else 30_000
GAMMA = 0.95
DQN_BATCH = 128
SYNC_EVERY = 500
MEMORY_SIZE = 50_000
EXPLORE_START, EXPLORE_END, EXPLORE_GAMES = 1.0, 0.1, 15_000
DQN_FORWARD_GAME = 300
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]


def winner(board):
    """+1 if X has three in a row, -1 if O does, else 0."""
    for i, j, k in LINES:
        if board[i] != 0 and board[i] == board[j] == board[k]:
            return board[i]
    return 0


def legal_moves(board):
    return [i for i in range(9) if board[i] == 0]


def play(board, square, player):
    return board[:square] + (player,) + board[square + 1:]


@lru_cache(maxsize=None)
def minimax_value(board, player):
    """Value for `player` to move with perfect play: +1, 0 or -1."""
    w = winner(board)
    if w:
        return w * player
    if 0 not in board:
        return 0
    return max(-minimax_value(play(board, i, player), -player) for i in legal_moves(board))


def perfect_moves(board, player):
    values = {i: -minimax_value(play(board, i, player), -player) for i in legal_moves(board)}
    best = max(values.values())
    return [i for i, v in values.items() if v == best]


def winning_squares(board, player):
    return [i for i in legal_moves(board) if winner(play(board, i, player)) == player]


def every_position():
    """Every board reachable in play that is not over, with the player to move."""
    positions = {}

    def walk(board, player):
        if winner(board) or 0 not in board or (board, player) in positions:
            return
        positions[(board, player)] = True
        for i in legal_moves(board):
            walk(play(board, i, player), -player)

    walk((0,) * 9, 1)
    return list(positions)


POSITIONS = every_position()
WIN_POSITIONS = [(b, p) for b, p in POSITIONS if winning_squares(b, p)]
BLOCK_POSITIONS = [(b, p) for b, p in POSITIONS if not winning_squares(b, p) and len(winning_squares(b, -p)) == 1]
# the forward-pass slide's position: O (-1) holds 0 and 3, so X (+1) must block at 6
TEACH = (-1, 1, 0,
         -1, 0, 0,
          0, 0, 1)
print(len(POSITIONS), "positions,", len(WIN_POSITIONS), "with an open win,", len(BLOCK_POSITIONS), "with one threat to block")

In [ ]:
def make_q_network():
    """9 board numbers (+1 mine, -1 theirs, 0 empty) in, one score per square out."""
    return nn.Sequential(nn.Linear(9, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 9))


def view(board, player):
    """The board as `player` sees it."""
    return torch.tensor([v * player for v in board], dtype=torch.float32)


def train_dqn():
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    q_net = make_q_network()
    frozen_copy = make_q_network()
    frozen_copy.load_state_dict(q_net.state_dict())
    optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)

    def q_hat(board, player, model=q_net):
        with torch.no_grad():
            return model(view(board, player)).tolist()

    def best_move(board, player):
        scores = q_hat(board, player)
        return max(legal_moves(board), key=lambda i: scores[i])

    def match(side, opponent):
        """One greedy game for the network on `side`: +1 win, 0 draw, -1 loss."""
        board, player = (0,) * 9, 1
        while True:
            move = best_move(board, player) if player == side else opponent(board, player)
            board = play(board, move, player)
            w = winner(board)
            if w:
                return 1 if w == side else -1
            if 0 not in board:
                return 0
            player = -player

    def random_player(board, player):
        return random.choice(legal_moves(board))

    def perfect_player(board, player):
        return random.choice(perfect_moves(board, player))

    result = {"curve": []}

    def record(game):
        take_win = np.mean([best_move(b, p) in winning_squares(b, p) for b, p in WIN_POSITIONS])
        block = np.mean([best_move(b, p) in winning_squares(b, -p) for b, p in BLOCK_POSITIONS])
        row = {"game": game, "take_win": float(take_win), "block": float(block)}
        for name, opponent in (("random", random_player), ("perfect", perfect_player)):
            outcomes = [match(side, opponent) for side in (1, -1) for _ in range(100)]
            row[name] = {"win": outcomes.count(1) / 200, "draw": outcomes.count(0) / 200, "loss": outcomes.count(-1) / 200}
        result["curve"].append(row)

    memory = []
    updates = 0
    record(0)
    for game in range(1, DQN_GAMES + 1):
        explore = max(EXPLORE_END, EXPLORE_START - (EXPLORE_START - EXPLORE_END) * game / EXPLORE_GAMES)
        board, player = (0,) * 9, 1
        while True:
            move = random.choice(legal_moves(board)) if random.random() < explore else best_move(board, player)
            next_board = play(board, move, player)
            won = bool(winner(next_board))
            done = won or 0 not in next_board
            reward = 1.0 if won else 0.0
            filled = [i not in legal_moves(next_board) for i in range(9)]
            memory.append((view(board, player), move, reward, view(next_board, -player), done, filled))
            if len(memory) > MEMORY_SIZE:
                memory.pop(0)
            if len(memory) >= DQN_BATCH:
                batch = random.sample(memory, DQN_BATCH)
                states = torch.stack([m[0] for m in batch])
                actions = torch.tensor([m[1] for m in batch])
                rewards = torch.tensor([m[2] for m in batch])
                next_states = torch.stack([m[3] for m in batch])
                finished = torch.tensor([m[4] for m in batch], dtype=torch.float32)
                taken = torch.tensor([m[5] for m in batch])
                with torch.no_grad():
                    next_best = frozen_copy(next_states).masked_fill(taken, -1e9).max(1).values
                    next_best = torch.where(finished > 0, torch.zeros_like(next_best), next_best)
                    target = rewards - GAMMA * next_best      # q = r - gamma * max Q(s')
                q_sa = q_net(states).gather(1, actions[:, None]).squeeze(1)
                loss = ((q_sa - target) ** 2).mean()
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                updates += 1
                if updates % SYNC_EVERY == 0:
                    frozen_copy.load_state_dict(q_net.state_dict())   # a total swap, not an average
            if done:
                break
            board, player = next_board, -player

        if game == DQN_FORWARD_GAME:
            after_block = play(TEACH, 6, 1)
            o_scores = q_hat(after_block, -1, frozen_copy)
            result["forward"] = {"Q": q_hat(TEACH, 1), "Q_next_opp": o_scores,
                                 "q2max": max(o_scores[i] for i in legal_moves(after_block))}
        if game % 1000 == 0 or (game <= 2000 and game % 100 == 0):
            record(game)
    return result


dqn = cached("dqn", train_dqn)

**Forward and backward pass (game 300).** X blocks at the bottom-left square. Slide: Q̂(s, a) = 0.29,
max Q̂(s′) = −0.51, q = 0.48, L = 0.04, ∂L/∂Q̂(s, a) = −0.38 (and 0 for the other eight squares).
As on the slide, each Q̂ and the target are rounded to two decimals before the next step.

In [ ]:
fwd = dqn["forward"]
q_sa = round(fwd["Q"][6], 2)
q2max = round(fwd["q2max"], 2)
target = round(-GAMMA * q2max, 2)
print("X's Q-hat on the slide's board:", np.round(fwd["Q"], 2))
print("O's Q-hat after the block:     ", np.round(fwd["Q_next_opp"], 2))
print(f"q = 0 - {GAMMA} x ({q2max:.2f}) = {target:.2f}")
print(f"L = ({q_sa:.2f} - {target:.2f})^2 = {(q_sa - target) ** 2:.2f}")
print(f"dL/dQ(s,a) = 2({q_sa:.2f} - {target:.2f}) = {2 * (q_sa - target):.2f}")

**Results.** Slide: takes an open win 92% of the time by game 200, blocks 95% by game 700;
at game 30,000 it wins 90% against a random player and draws 100% against a perfect one.

In [ ]:
curve = dqn["curve"]
games = [c["game"] for c in curve]
plt.plot(games, [c["take_win"] for c in curve], color=BLUE, label="takes an open win")
plt.plot(games, [c["block"] for c in curve], color=ORANGE, label="blocks a threat")
plt.plot(games, [c["perfect"]["draw"] for c in curve], color=GREY, label="draws vs a perfect player")
plt.xscale("symlog", linthresh=100)
plt.xlabel("games of self-play")
plt.ylabel("fraction")
plt.title("Skills arrive in order: win first, then block")
plt.legend()
plt.show()

by_game = {c["game"]: c for c in curve}
for game in (200, 700, DQN_GAMES):
    if game in by_game:
        c = by_game[game]
        print(f"game {game:>6,}: take win {c['take_win']:.0%}  block {c['block']:.0%}  "
              f"vs random win {c['random']['win']:.0%}  vs perfect draw {c['perfect']['draw']:.0%}")

## 3. Iterative refinement: diffusion
Slide: *Diffusion: destroy a little at a time*. x_t = √ᾱ_t · x₀ + √(1 − ᾱ_t) · ε with the DDPM schedule
(β from 10⁻⁴ to 0.02 over T = 1,000 steps). The weights left on the image and on the noise depend only on the schedule.
The slide's 7 is hand-drawn; here it is a 7 from the dataset.

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

seven = DIGITS[LABELS == 7][0].view(28, 28) * 2 - 1
noise_rng = np.random.default_rng(0)
images, titles = [], []
for t in [0, 50, 150, 300, 600, 1000]:
    if t == 0:
        a_bar = 1.0
        x_t = seven
    else:
        a_bar = alpha_bar[t - 1].item()
        eps = torch.tensor(noise_rng.standard_normal((28, 28)), dtype=torch.float32)
        x_t = math.sqrt(a_bar) * seven + math.sqrt(1 - a_bar) * eps
    images.append(x_t.clamp(-1, 1))
    titles.append(f"t = {t}\nimage ×{math.sqrt(a_bar):.2f}  noise ×{math.sqrt(1 - a_bar):.2f}")
show_images(images, titles, "Forward diffusion: no learning in this direction", cmap="gray", size=2.2)

Slide: *Learn one step, then loop it* (top rows). A small MLP learns to predict the noise ε in x_t
(loss ‖ε − ε̂‖²). One denoise step at t = 400 subtracts a little of ε̂: x_{t−1} = (x_t − 0.009 · ε̂) ÷ 0.996.
30,000 steps, about 5 minutes.

In [ ]:
DIFF_STEPS = 500 if QUICK else 30_000
T_SHOW = 400


def time_embedding(t, dim=64):
    """Sinusoidal embedding of the timestep, like position encoding in a transformer."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    angles = t.float()[:, None] * freqs[None]
    return torch.cat([angles.sin(), angles.cos()], dim=1)


class MLPDenoiser(nn.Module):
    """[784 pixels, 64-number embedding of t] -> 3 hidden layers of 512 -> predicted noise (784)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784 + 64, 512), nn.SiLU(), nn.Linear(512, 512), nn.SiLU(),
                                 nn.Linear(512, 512), nn.SiLU(), nn.Linear(512, 784))

    def forward(self, x, t):
        return self.net(torch.cat([x, time_embedding(t)], dim=1))


def train_denoise_step():
    torch.manual_seed(0)
    np.random.seed(0)
    X = DIGITS * 2 - 1
    model = MLPDenoiser()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, DIFF_STEPS)
    for step in range(1, DIFF_STEPS + 1):
        x0 = X[torch.randint(0, len(X), (128,))]
        t = torch.randint(0, T, (128,))
        eps = torch.randn_like(x0)
        x_t = alpha_bar[t].sqrt()[:, None] * x0 + (1 - alpha_bar[t]).sqrt()[:, None] * eps
        loss = ((model(x_t, t) - eps) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        schedule.step()

    model.eval()
    generator = torch.Generator().manual_seed(0)
    with torch.no_grad():
        x0 = X[7][None]                                  # one digit from the dataset (index 7)
        t = torch.tensor([T_SHOW])
        eps = torch.randn(1, 784, generator=generator)
        x_t = alpha_bar[t].sqrt() * x0 + (1 - alpha_bar[t]).sqrt() * eps
        eps_hat = model(x_t, t)
        c = (betas[t] / (1 - alpha_bar[t]).sqrt()).item()
        x_prev = (x_t - c * eps_hat) / alphas[t].sqrt()
    pictures = {"x0": x0, "x_t": x_t, "eps": eps, "eps_hat": eps_hat, "x_prev": x_prev}
    return {"c": c, "sqrt_alpha": alphas[T_SHOW].sqrt().item(), "final_loss": loss.item(),
            "eps_mse": ((eps - eps_hat) ** 2).mean().item(),
            "pictures": {k: v.view(28, 28).tolist() for k, v in pictures.items()}}


step = cached("diffusion_denoise_step", train_denoise_step)
print(f"x_(t-1) = (x_t - {step['c']:.3f} * eps_hat) / {step['sqrt_alpha']:.3f}    (t = {T_SHOW})")
print(f"training loss {step['final_loss']:.3f}; this example's ||eps - eps_hat||^2 per pixel {step['eps_mse']:.2f}")
pics = step["pictures"]
show_images([np.clip(pics[k], -1, 1) if "eps" not in k else np.clip(pics[k], -2.5, 2.5) for k in ["x0", "x_t", "eps", "eps_hat", "x_prev"]],
            ["x₀", f"x_t (t = {T_SHOW})", "ε (true)", "ε̂ (predicted)", "x_(t−1)"],
            "Predict the noise, subtract a sliver of it", size=2.0)

Slide: *Learn one step, then loop it* (bottom row). The MLP above is too small to make good digits from pure noise,
so the loop row came from a small CNN denoiser (a U-Net-like shape: two stride-2 steps down, two back up, with skips)
trained for 10 epochs on all 60,000 MNIST training digits on a Colab T4 (about 10 minutes).
On a CPU this cell takes hours: use a GPU (Runtime → Change runtime type → T4 GPU) or keep `QUICK = True`.
GPU runs are not bit-for-bit reproducible, so your digits will differ from the slide's.

In [ ]:
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CNN_EPOCHS = 10
CNN_BATCHES_QUICK = 20


class Block(nn.Module):
    """Two 3x3 convolutions with SiLU; the time vector is added in between."""
    def __init__(self, c_in, c_out, t_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.time = nn.Linear(t_dim, c_out)
        self.norm1 = nn.GroupNorm(8, c_out)
        self.norm2 = nn.GroupNorm(8, c_out)

    def forward(self, x, t_vec):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.time(t_vec)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h)))


class CNNDenoiser(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.down1 = Block(1, c)                                        # 28 x 28
        self.pool1 = nn.Conv2d(c, c, 3, stride=2, padding=1)            # -> 14 x 14
        self.down2 = Block(c, 2 * c)
        self.pool2 = nn.Conv2d(2 * c, 2 * c, 3, stride=2, padding=1)    # -> 7 x 7
        self.mid = Block(2 * c, 2 * c)
        self.up2 = nn.ConvTranspose2d(2 * c, 2 * c, 4, stride=2, padding=1)   # -> 14 x 14
        self.dec2 = Block(4 * c, c)
        self.up1 = nn.ConvTranspose2d(c, c, 4, stride=2, padding=1)           # -> 28 x 28
        self.dec1 = Block(2 * c, c)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x, t):
        t_vec = time_embedding(t)
        h1 = self.down1(x, t_vec)
        h2 = self.down2(self.pool1(h1), t_vec)
        m = self.mid(self.pool2(h2), t_vec)
        d2 = self.dec2(torch.cat([self.up2(m), h2], dim=1), t_vec)
        d1 = self.dec1(torch.cat([self.up1(d2), h1], dim=1), t_vec)
        return self.out(d1)


def train_cnn_and_generate():
    torch.manual_seed(0)
    mnist = torchvision.datasets.MNIST("data", train=True, download=True, transform=torchvision.transforms.ToTensor())
    loader = torch.utils.data.DataLoader(mnist, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
    model = CNNDenoiser().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    b, a, ab = betas.to(device), alphas.to(device), alpha_bar.to(device)
    model.train()
    for epoch in range(1 if QUICK else CNN_EPOCHS):
        for batch_index, (x0, _) in enumerate(loader):
            if QUICK and batch_index == CNN_BATCHES_QUICK:
                break
            x0 = x0.to(device) * 2 - 1
            t = torch.randint(0, T, (x0.shape[0],), device=device)
            eps = torch.randn_like(x0)
            x_t = ab[t].sqrt()[:, None, None, None] * x0 + (1 - ab[t]).sqrt()[:, None, None, None] * eps
            loss = F.mse_loss(model(x_t, t), eps)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"epoch {epoch + 1}: loss {loss.item():.4f}")

    # generate: loop the one denoise step from t = 999 down to 0
    model.eval()
    snaps = [999, 750, 500, 300, 150, 0]
    frames = {}
    with torch.no_grad():
        x = torch.randn(6, 1, 28, 28, device=device)
        for t in range(T - 1, -1, -1):
            eps_hat = model(x, torch.full((x.shape[0],), t, device=device))
            x = (x - b[t] / (1 - ab[t]).sqrt() * eps_hat) / a[t].sqrt()
            if t > 0:
                x = x + b[t].sqrt() * torch.randn_like(x)      # sigma_t z: fresh noise, except at the last step
            if t in snaps:
                frames[str(t)] = x.clamp(-1, 1)[:, 0].cpu().tolist()
    return frames


frames = cached("diffusion_cnn_frames", train_cnn_and_generate)
order = [str(t) for t in [999, 750, 500, 300, 150, 0]]
fig, axes = plt.subplots(3, len(order), figsize=(1.6 * len(order), 5))
for col, t in enumerate(order):
    for row in range(3):
        axes[row, col].imshow(frames[t][row], cmap="gray")
        axes[row, col].axis("off")
    axes[0, col].set_title(f"t = {t}", fontsize=9)
fig.suptitle("Loop the step 1,000 times: noise becomes a digit")
plt.show()

## 4. Rendering: a tiny NeRF
Slides: *Render a pixel, then a photo*, *NeRF: the structure*, *Results: a view no camera took*.
The scene is two shaded spheres we render ourselves, photographed by 24 cameras on a circle (32 × 32 pixels each);
one more camera halfway between two of them is held out. Every pixel is one example: its ray in, its colour as the label.
The MLP maps a point (x, y, z), encoded with sin/cos at 6 frequencies (39 numbers), to a density σ and a colour c.
The renderer blends the 16 points on each ray with weights wᵢ computed from σ (no learned weights in the renderer).
3,000 steps of 1,024 rays, about 15 minutes; the same run without the encoding takes another 15.

In [ ]:
NERF_SIZE = 32
FOCAL = 36.0
N_CAMERAS = 24
N_POINTS = 16
NEAR, FAR = 2.0, 6.0
NERF_STEPS = 300 if QUICK else 3000
NERF_BATCH = 1024
SPHERES = [((-0.45, 0.0, 0.0), 0.75, (0.93, 0.45, 0.10)),     # centre, radius, colour
           ((0.6, 0.15, -0.35), 0.55, (0.15, 0.35, 0.85))]
LIGHT = np.array([0.5, 0.8, 0.6]) / np.linalg.norm([0.5, 0.8, 0.6])


def camera(angle_deg, elev_deg=20.0, radius=4.0):
    """Camera position and its forward, right and up directions, looking at the origin."""
    a, e = math.radians(angle_deg), math.radians(elev_deg)
    origin = radius * np.array([math.cos(e) * math.sin(a), math.sin(e), math.cos(e) * math.cos(a)])
    forward = -origin / np.linalg.norm(origin)
    right = np.cross(forward, [0, 1, 0])
    right /= np.linalg.norm(right)
    up = np.cross(right, forward)
    return origin, forward, right, up


def camera_rays(angle_deg):
    """Origin and unit direction of the ray through every pixel."""
    origin, forward, right, up = camera(angle_deg)
    cols, rows = np.meshgrid(np.arange(NERF_SIZE), np.arange(NERF_SIZE))
    x = (cols + 0.5 - NERF_SIZE / 2) / FOCAL
    y = -(rows + 0.5 - NERF_SIZE / 2) / FOCAL
    directions = forward[None, None] + x[..., None] * right + y[..., None] * up
    directions /= np.linalg.norm(directions, axis=-1, keepdims=True)
    return np.broadcast_to(origin, directions.shape).copy(), directions


def photograph(angle_deg):
    """The true photo: exact ray-sphere hits, Lambert shading, white background."""
    origins, directions = camera_rays(angle_deg)
    image = np.ones(origins.shape)
    nearest = np.full(origins.shape[:2], np.inf)
    for centre, radius, colour in SPHERES:
        to_ray = origins - np.array(centre)
        b = (to_ray * directions).sum(-1)
        c = (to_ray * to_ray).sum(-1) - radius * radius
        disc = b * b - c
        hit = disc > 0
        t = -b - np.sqrt(np.where(hit, disc, 0))
        hit &= (t > 0) & (t < nearest)
        normal = (origins + t[..., None] * directions - np.array(centre)) / radius
        shade = 0.35 + 0.65 * np.clip((normal * LIGHT).sum(-1), 0, 1)
        image[hit] = np.array(colour)[None] * shade[hit][:, None]
        nearest[hit] = t[hit]
    return image


TRAIN_ANGLES = [i * 360 / N_CAMERAS for i in range(N_CAMERAS)]
HELD_OUT_ANGLE = 7.5 * 360 / N_CAMERAS + 360 / N_CAMERAS / 2       # halfway between two training cameras
PHOTOS = [photograph(a) for a in TRAIN_ANGLES]
RAY_O = torch.tensor(np.concatenate([camera_rays(a)[0].reshape(-1, 3) for a in TRAIN_ANGLES]), dtype=torch.float32)
RAY_D = torch.tensor(np.concatenate([camera_rays(a)[1].reshape(-1, 3) for a in TRAIN_ANGLES]), dtype=torch.float32)
PIXELS = torch.tensor(np.concatenate([p.reshape(-1, 3) for p in PHOTOS]), dtype=torch.float32)
HELD_OUT_PHOTO = photograph(HELD_OUT_ANGLE)
print(len(PIXELS), "labelled pixels from", N_CAMERAS, "photos")
show_images(PHOTOS[::4], [f"camera {i + 1}" for i in range(0, N_CAMERAS, 4)], "Some of the training photos", cmap=None)

In [ ]:
def positional_encoding(x, n_freqs):
    """x plus sin and cos of 2^k * pi * x for k = 0 .. n_freqs - 1 (3 + 6 * n_freqs numbers)."""
    parts = [x]
    for k in range(n_freqs):
        parts += [torch.sin((2 ** k) * math.pi * x), torch.cos((2 ** k) * math.pi * x)]
    return torch.cat(parts, -1)


def render_rays(model, n_freqs, origins, directions, stratified):
    """Volume rendering: c_hat = sum_i w_i c_i + (1 - sum_i w_i) * white, w_i from the densities."""
    n = origins.shape[0]
    edges = torch.linspace(NEAR, FAR, N_POINTS + 1)
    if stratified:
        t = edges[:-1] + (edges[1:] - edges[:-1]) * torch.rand(n, N_POINTS)   # a random spot in each slice
    else:
        t = ((edges[:-1] + edges[1:]) / 2).expand(n, N_POINTS)
    points = origins[:, None] + t[..., None] * directions[:, None]
    raw = model(positional_encoding(points / 2.0, n_freqs))                  # scene scaled into about [-1, 1]
    sigma = F.softplus(raw[..., 0])
    colour = torch.sigmoid(raw[..., 1:])
    delta = torch.cat([t[:, 1:] - t[:, :-1], torch.full((n, 1), (FAR - NEAR) / N_POINTS)], -1)
    alpha = 1 - torch.exp(-sigma * delta)
    transmittance = torch.cumprod(torch.cat([torch.ones(n, 1), 1 - alpha[:, :-1] + 1e-10], -1), -1)
    w = transmittance * alpha
    return (w[..., None] * colour).sum(1) + (1 - w.sum(1, keepdim=True))


def render_photo(model, n_freqs, angle_deg):
    origins, directions = camera_rays(angle_deg)
    origins = torch.tensor(origins.reshape(-1, 3), dtype=torch.float32)
    directions = torch.tensor(directions.reshape(-1, 3), dtype=torch.float32)
    with torch.no_grad():
        image = render_rays(model, n_freqs, origins, directions, stratified=False)
    return image.reshape(NERF_SIZE, NERF_SIZE, 3).numpy()


def psnr(image, truth):
    """Peak signal-to-noise ratio in dB: -10 log10(MSE) for pixels in [0, 1]. Higher is better."""
    return float(-10 * np.log10(((image - truth) ** 2).mean()))


def train_nerf(n_freqs):
    torch.manual_seed(0)
    np.random.seed(0)
    model = nn.Sequential(nn.Linear(3 + 6 * n_freqs, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(),
                          nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 4))
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
    result = {"psnr": {}, "held_out": {}, "n_weights": sum(p.numel() for p in model.parameters())}
    for step in range(NERF_STEPS + 1):
        if step in (0, 100, 300, 1000, 3000):
            image = render_photo(model, n_freqs, HELD_OUT_ANGLE)
            result["psnr"][str(step)] = psnr(image, HELD_OUT_PHOTO)
            result["held_out"][str(step)] = image.tolist()
        if step == 300:
            result["camera_1_step_300"] = render_photo(model, n_freqs, TRAIN_ANGLES[0]).tolist()
        if step == NERF_STEPS:
            break
        batch = torch.randint(0, len(PIXELS), (NERF_BATCH,))
        loss = ((render_rays(model, n_freqs, RAY_O[batch], RAY_D[batch], stratified=True) - PIXELS[batch]) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return result


nerf = cached("nerf", lambda: train_nerf(n_freqs=6))

**Render a pixel, then a photo (step 300).** Slide: the pixel in row 17, column 13 of camera 1 renders as
ĉ = (0.82, 0.42, 0.02) against the photo's c = (0.80, 0.39, 0.09); the whole predicted photo has MSE 0.0033.

In [ ]:
if "camera_1_step_300" in nerf:
    predicted = np.array(nerf["camera_1_step_300"])
    row, col = 16, 12
    print("c_hat =", np.round(predicted[row, col], 2), "  c =", np.round(PHOTOS[0][row, col], 2))
    print(f"MSE over the photo: {((predicted - PHOTOS[0]) ** 2).mean():.4f}")
    show_images([np.clip(predicted, 0, 1), PHOTOS[0]], ["predicted, step 300", "the photo (label)"], "Camera 1", cmap=None, size=2.4)

**Results.** Slide: held-out PSNR 6.8, 22.7, 23.3, 29.4 and 35.3 dB at steps 0, 100, 300, 1,000 and 3,000,
with 38,660 weights.

In [ ]:
steps = list(nerf["held_out"])
print("weights:", nerf["n_weights"])
print("held-out PSNR (dB):", {int(s): round(v, 1) for s, v in nerf["psnr"].items()})
show_images([np.clip(nerf["held_out"][s], 0, 1) for s in steps] + [HELD_OUT_PHOTO],
            [f"step {int(s):,}\n{nerf['psnr'][s]:.1f} dB" for s in steps] + ["true view"],
            "The held-out view: no camera took it", cmap=None, size=2.0)

**Why the sin/cos encoding (structure slide).** The same run on the raw (x, y, z), no encoding.
Slide: 25.7 dB raw vs 35.3 dB with γ.

In [ ]:
nerf_raw = cached("nerf_no_encoding", lambda: train_nerf(n_freqs=0))
last = str(NERF_STEPS)
print(f"held-out PSNR at step {NERF_STEPS:,}: raw (x, y, z) {nerf_raw['psnr'][last]:.1f} dB, "
      f"with the encoding {nerf['psnr'][last]:.1f} dB")
show_images([np.clip(nerf_raw["held_out"][last], 0, 1), np.clip(nerf["held_out"][last], 0, 1), HELD_OUT_PHOTO],
            ["raw (x, y, z)", "with sin/cos encoding", "true view"], "Without the encoding the MLP only draws blurs",
            cmap=None, size=2.4)

## 5. Your turn: the invention of writing
Slide: *One way to build it*. A carver (a small CNN) sees a picture and carves 3 marks, each one of 10 shapes
(softmax, then argmax). A reader (an MLP) sees only the marks and names the picture; the loss is cross-entropy.
The digit classes stand in for the animals. argmax has no slope, so the carver can't learn from the reader
unless we fake one: forward uses argmax, backward uses the softmax's gradient (`hard + soft - soft.detach()`).
3,000 steps per run, a few minutes. Accuracy is on 1,000 held-out digits; chance is 10%.

In [ ]:
MARKS, SHAPES, CLASSES = 3, 10, 10
WRITE_STEPS = 300 if QUICK else 3000
SHAPE_GLYPHS = "|○△□✕∿∧⌒‖⋯"

images = DIGITS.view(-1, 1, 28, 28)
split = torch.randperm(len(images), generator=torch.Generator().manual_seed(0))
train_x, train_y = images[split[1000:]], LABELS[split[1000:]]
test_x, test_y = images[split[:1000]], LABELS[split[:1000]]


class Carver(nn.Module):
    """Picture -> logits for each of the 3 marks over the 10 shapes."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.SiLU(), nn.MaxPool2d(2),
                                  nn.Conv2d(16, 32, 3, padding=1), nn.SiLU(), nn.MaxPool2d(2))
        self.head = nn.Linear(32 * 7 * 7, MARKS * SHAPES)

    def forward(self, x):
        return self.head(self.conv(x).flatten(1)).view(-1, MARKS, SHAPES)


def carve(logits, surrogate):
    """The carved marks as one-hot shapes. With surrogate=True the backward pass uses the softmax's gradient."""
    soft = F.softmax(logits, dim=-1)
    hard = F.one_hot(soft.argmax(-1), SHAPES).float()
    if not surrogate:
        return hard                        # argmax: no slope, nothing reaches the carver
    return hard + soft - soft.detach()     # forward value = hard, gradient = the softmax's


def train_writing(surrogate):
    torch.manual_seed(0)
    carver = Carver()
    reader = nn.Sequential(nn.Linear(MARKS * SHAPES, 128), nn.SiLU(), nn.Linear(128, CLASSES))
    optimizer = torch.optim.Adam(list(carver.parameters()) + list(reader.parameters()), lr=1e-3)
    curve = []
    for step in range(1, WRITE_STEPS + 1):
        batch = torch.randint(0, len(train_x), (128,))
        marks = carve(carver(train_x[batch]), surrogate)
        loss = F.cross_entropy(reader(marks.flatten(1)), train_y[batch])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 250 == 0 or step == 1:
            with torch.no_grad():
                test_marks = F.one_hot(carver(test_x).argmax(-1), SHAPES).float()
                accuracy = (reader(test_marks.flatten(1)).argmax(1) == test_y).float().mean().item()
            curve.append({"step": step, "accuracy": accuracy})
    with torch.no_grad():
        messages = carver(test_x).argmax(-1)
    words = {}
    for digit in range(CLASSES):
        rows = [tuple(m) for m in messages[test_y == digit].tolist()]
        words[digit] = "".join(SHAPE_GLYPHS[s] for s in max(set(rows), key=rows.count))
    return {"curve": curve, "words": words}


writing = {
    "argmax, no gradient": cached("writing_blocked", lambda: train_writing(surrogate=False)),
    "surrogate gradient": cached("writing_surrogate", lambda: train_writing(surrogate=True)),
}

Slide: 25% with no gradient through argmax, 96% with the surrogate. Words it invented: ✕|⋯ for 0, ✕∧| for 2, |∧△ for 3, ||△ for 5, |□⋯ for 9.

In [ ]:
for (name, run), colour in zip(writing.items(), [GREY, BLUE]):
    steps = [c["step"] for c in run["curve"]]
    plt.plot(steps, [c["accuracy"] for c in run["curve"]], color=colour, label=name)
    print(f"{name}: held-out accuracy {run['curve'][-1]['accuracy']:.1%}")
plt.axhline(0.1, color=INK, linestyle=":", label="chance")
plt.xlabel("training step")
plt.ylabel("held-out accuracy")
plt.title("The carver only learns through a surrogate gradient")
plt.legend()
plt.show()

print("Most common word for each digit (surrogate run):")
for digit, word in writing["surrogate gradient"]["words"].items():
    print(f"  {digit}: {word}")